## Cleaning

### Data Import

In [ ]:
import polars as pl
import altair as alt
from dotenv import load_dotenv
import os

load_dotenv(override=True)  # override=True ensures .env values take precedence over system env vars

USER = os.getenv("USER")
PASSWORD = os.getenv("PW")
HOST = os.getenv("HOST")
DATABASE = os.getenv("DB_NAME")

uri = f"postgresql://{USER}:{PASSWORD}@{HOST}:5432/{DATABASE}"

query = "SELECT * FROM daan_822.article_detailed"

df = pl.read_database_uri(query=query, uri=uri)

In [ ]:
df.describe()

### Cleaning
- Strip leading/trailing whitespace from all string columns
- Replace empty strings with null across all string columns
- Cast copyright_year, figure_count, table_count from Text to Int16
- Parse pub_date string to Date type with multi-format fallback (YYYY-MM-DD, YYYY-MM, YYYY)
- Drop license_type column (entirely null)
- Lowercase keywords, journal_title, publisher_name, article_type, article_subject
- Parse authors JSON string into a typed struct list
- Parse keywords JSON string into a string list
- Detect remaining JSON array columns and convert to comma-separated strings
- Explode authors into df_authors; strip whitespace, null out empty names, validate ORCID format (^\d{4}-\d{4}-\d{4}-\d{3}[\dX]$), build author_full_name
- Explode keywords into df_keyword

In [ ]:
# cleaning whitespace and replacing blank with true nulls
df_cleaning = df.with_columns(pl.selectors.string().str.strip_chars().replace("", None))

# converting string type to int
df_cleaning = df_cleaning.with_columns(
    pl.selectors.by_name("copyright_year", "figure_count", "table_count").cast(pl.Int16)
)

# converting pub_date to date
df_cleaning = df_cleaning.with_columns(
    pl.coalesce(
        [
            pl.col("pub_date").str.to_date("%Y-%m-%d", strict=False),
            pl.col("pub_date").str.to_date("%Y-%m", strict=False),
            pl.col("pub_date").str.to_date("%Y", strict=False),
        ]
    )
)

# dropping column as it is null
df_cleaning = df_cleaning.drop("license_type")

# making columns lower case
df_cleaning = df_cleaning.with_columns(
    pl.selectors.by_name("keywords", "journal_title", "publisher_name", "article_type", "article_subject").str.to_lowercase()
)

In [ ]:
df_cleaning.describe()

In [ ]:
# labelling field names in the json construct for authors
dtypes = pl.List(
    pl.Struct(
        [
            pl.Field("given_names", pl.String),
            pl.Field("is_corresponding", pl.Boolean),
            pl.Field("orcid", pl.String),
            pl.Field("surname", pl.String),
        ]
    )
)

# parse JSON columns without exploding to avoid author x keyword cartesian product
df_parsed = df_cleaning.with_columns(
    pl.col("authors").str.json_decode(dtypes).alias("authors_struct"),
    pl.col("keywords").str.json_decode(pl.List(pl.String)).alias("keywords_list"),
).drop("authors", "keywords")

# identify columns that are json formatted in order to clean to be comma separated
json_array_cols = [
    col
    for col, dtype in df_parsed.schema.items()
    if dtype == pl.String
    and df_parsed.select(pl.col(col).drop_nulls().str.starts_with("[").all()).item()
]

# clean up JSON array columns to comma-separated strings
df_base = df_parsed.with_columns(
    [
        pl.col(col).str.json_decode(pl.List(pl.String)).list.join(", ").alias(col)
        for col in json_array_cols
    ]
)

# separate exploded views to avoid author x keyword cartesian product
df_authors = (
    df_base.explode("authors_struct")
    .unnest("authors_struct")
    .with_columns(
        # strip whitespace from name fields
        pl.col("given_names").str.strip_chars(),
        pl.col("surname").str.strip_chars(),
        pl.col("orcid").str.strip_chars(),
    )
    .with_columns(
        # replace empty strings with null
        pl.col("given_names").replace("", None),
        pl.col("surname").replace("", None),
        pl.col("orcid").replace("", None),
    )
    .with_columns(
        # validate ORCID format — set invalid values to null
        pl.when(pl.col("orcid").str.contains(r"^\d{4}-\d{4}-\d{4}-\d{3}[\dX]$"))
        .then(pl.col("orcid"))
        .otherwise(None)
        .alias("orcid")
    )
    .with_columns(
        (pl.col("given_names").fill_null("") + " " + pl.col("surname").fill_null(""))
        .str.strip_chars()
        .replace("", None)
        .alias("author_full_name")
    )
)

df_keywords = df_base.explode("keywords_list").rename({"keywords_list": "keyword"})

In [ ]:
df_base.describe()

#### Missingness Handler

In [ ]:
total_rows = df_base.height

cols_with_null_info = [
    (col, df_base.select(pl.col(col).null_count()).item(),
     round(df_base.select(pl.col(col).null_count()).item() / total_rows * 100, 2), df_base.schema[col])
    for col in df_base.columns
    if df_base.select(pl.col(col).null_count()).item() > 0
]

# how many rows are missing copyright_year but have a pub_date?
missing_cy = df_base.filter(pl.col("copyright_year").is_null())

print(f"Missing copyright_year: {missing_cy.height}")
print(f"  - with pub_date available: {missing_cy.filter(pl.col('pub_date').is_not_null()).height}")
print(f"  - without pub_date:        {missing_cy.filter(pl.col('pub_date').is_null()).height}")

df_base = df_base.with_columns(
    pl.when(pl.col("copyright_year").is_null())
    .then(pl.col("pub_date").dt.year().cast(pl.Int16))
    .otherwise(pl.col("copyright_year"))
    .alias("copyright_year")
)

#### Copyright Year validation

In [ ]:
# extracts year from the copyright statement
df_base = df_base.with_columns(
    pl.col("copyright_statement")
    .str.extract(r"((?:19|20)\d{2})")
    .cast(pl.Int16, strict=False)
    .alias("year_from_statement")
)

# compares the result
df_base = df_base.with_columns(
    (pl.col("copyright_year") == pl.col("year_from_statement")).alias("copyright_year_match")
)

# if the year extracted is not null, it replaces the copyright year
df_base = df_base.with_columns(
    pl.when(pl.col("year_from_statement").is_not_null())
    .then(pl.col("year_from_statement"))
    .otherwise(pl.col("copyright_year"))
    .alias("copyright_year")
)

# there are only 25 rows that did not match when compared
copyright_year_mismatch_df = df_base.filter(pl.col("copyright_year_match") == False)

df_base = df_base.with_columns(
    pl.col("pub_date").dt.year().cast(pl.Int16).alias("pub_year")
).with_columns(
    (pl.col("copyright_year") == pl.col("pub_year")).alias("copyright_eq_pub")
)

#### Cleaning Effectiveness Summary

In [ ]:
# --- before/after null counts ---
raw_nulls = {col: df.select(pl.col(col).null_count()).item() for col in df.columns}
# for df_base, map cleaned columns; license_type was dropped
cleaned_nulls = {col: df_base.select(pl.col(col).null_count()).item() for col in df.columns if col in df_base.columns}

# --- whitespace / empty-string counts in raw data ---
raw_empty = {
    col: df.select(
        (pl.col(col).str.strip_chars() == "").sum()
    ).item()
    for col in df.columns if df.schema[col] == pl.String
}

rows = []

# 1. Whitespace & empty-string → null normalization
total_empty = sum(raw_empty.values())
rows.append(("Whitespace / empty-string → null", f"{total_empty} empty strings across {sum(1 for v in raw_empty.values() if v > 0)} columns", "0 empty strings remain", "All converted to proper nulls"))

# 2. Type casts (copyright_year, figure_count, table_count)
for col in ["copyright_year", "figure_count", "table_count"]:
    rows.append((f"Cast {col} to Int16", f"Type: {df.schema[col]}", f"Type: {df_base.schema[col]}", "Enables numeric operations"))

# 3. pub_date parsing
raw_pub_null = raw_nulls["pub_date"]
cleaned_pub_null = cleaned_nulls["pub_date"]
rows.append(("Parse pub_date → Date", f"Type: {df.schema['pub_date']}, {raw_pub_null} nulls", f"Type: {df_base.schema['pub_date']}, {cleaned_pub_null} nulls", "Multi-format date parsing"))

# 4. Drop license_type
rows.append(("Drop license_type", f"{raw_nulls['license_type']}/{df.height} nulls ({round(raw_nulls['license_type']/df.height*100,1)}%)", "Column removed", "Entirely null column dropped"))

# 5. Lowercasing
for col in ["keywords", "journal_title", "publisher_name", "article_type", "article_subject"]:
    raw_mixed = df.select(
        pl.col(col).drop_nulls()
        .str.strip_chars()
        .filter(pl.col(col).str.strip_chars() != pl.col(col).str.strip_chars().str.to_lowercase())
        .len()
    ).item()
    rows.append((f"Lowercase {col}", f"{raw_mixed} rows with mixed case", "All lowercase", "Consistent casing"))

# 6. copyright_year missingness recovery
raw_cy_null = raw_nulls["copyright_year"]
cleaned_cy_null = cleaned_nulls["copyright_year"]
rows.append(("Fill copyright_year from pub_date", f"{raw_cy_null} nulls", f"{cleaned_cy_null} nulls", f"{raw_cy_null - cleaned_cy_null} values recovered"))

# 7. Copyright year validation
mismatch_count = copyright_year_mismatch_df.height
rows.append(("Copyright year validation", f"{mismatch_count} year mismatches vs statement", "Corrected from copyright_statement", "Statement year preferred when available"))

# 8. ORCID validation
raw_orcid_non_null = df_authors.height  # total author rows
invalid_orcid = df_authors.select(
    pl.col("orcid").is_null().sum()
).item()
rows.append(("ORCID format validation", "Unvalidated ORCIDs", f"{invalid_orcid} null (invalid or missing)", "Regex-validated format"))

# 9. JSON parsing (authors, keywords)
rows.append(("Parse authors JSON → struct list", "Raw JSON string", f"List(Struct) with {df_authors.select(pl.col('author_full_name').drop_nulls().n_unique()).item()} unique authors", "Structured author data"))
rows.append(("Parse keywords JSON → string list", "Raw JSON string", f"List(String) with {df_keywords.select(pl.col('keyword').drop_nulls().n_unique()).item()} unique keywords", "Structured keyword data"))

cleaning_summary = pl.DataFrame(
    {
        "Cleaning Task": [r[0] for r in rows],
        "Before": [r[1] for r in rows],
        "After": [r[2] for r in rows],
        "Impact": [r[3] for r in rows],
    }
)

cleaning_summary

## Analysis

In [ ]:
df_base.schema

In [ ]:
df.describe()

In [ ]:
# creating a summary dataframe of common metrics
summary_df = pl.DataFrame(
    {
        "Metric": [
            "Total Articles",
            "Unique Authors",
            "Unique Keywords",
            "Avg Authors per Paper",
            "Avg Keywords per Paper",
        ],
        "Value": [
            df_base.select(pl.col("pmid").n_unique()).item(),
            df_authors.select(pl.col("author_full_name").n_unique()).item(),
            df_keywords.select(pl.col("keyword").n_unique()).item(),
            df_authors.group_by("pmid")
            .agg(pl.col("author_full_name").n_unique().alias("n_authors"))
            .select(pl.col("n_authors").mean().round(2))
            .item(),
            df_keywords.group_by("pmid")
            .agg(pl.col("keyword").n_unique().alias("n_keywords"))
            .select(pl.col("n_keywords").mean().round(2))
            .item(),
        ],
    },
    strict=False,
)

summary_df

#### Building Visuals

In [ ]:
data_avail = df_base.group_by("data_available").agg(pl.len().alias("count"))

data_avail

In [ ]:
pub_year_counts = (
    df_base.with_columns(pl.col("pub_date").dt.year().alias("pub_year"))
    .group_by("pub_year")
    .agg(pl.col("pmid").n_unique().alias("num_articles"))
    .sort("pub_year")
)

pub_year_counts

In [ ]:
pub_month_counts = (
    df_base
    .with_columns(
        pl.col("pub_date")
          .dt.truncate("1mo")
          .alias("pub_month")
    )
    .group_by("pub_month")
    .agg(pl.col("pmid").n_unique().alias("num_articles"))
    .sort("pub_month")
)

pub_month_counts

In [ ]:
top_keywords = (
    df_keywords
    .filter(pl.col("keyword").is_not_null())
    .group_by("keyword")
    .agg(pl.col("pmid").n_unique().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

top_keywords

In [ ]:
top_authors = (
    df_authors
    .filter(pl.col("author_full_name").is_not_null())
    .group_by("author_full_name")
    .agg(pl.col("pmid").n_unique().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

top_authors

#### Visualizing

In [ ]:
(
    alt.Chart(data_avail)
    .mark_bar()
    .encode(
        x=alt.X("data_available:O", title=""),
        y=alt.Y("count:Q", title="Number of Articles"),
    )
    .properties(title="Data Availability")
)

In [ ]:
(
    alt.Chart(pub_year_counts)
    .mark_bar()
    .encode(
        x=alt.X("pub_year:O", title="Publication Year"),
        y=alt.Y("num_articles:Q", title="Number of Articles"),
    )
    .properties(title="Publications per Year")
)

In [ ]:
(
    alt.Chart(pub_month_counts)
    .mark_line(point=True)
    .encode(
        x=alt.X("pub_month:T", title="Publication Date"),
        y=alt.Y("num_articles:Q", title="Number of Articles")
    )
    .properties(title="Publications Over Time")
)

In [ ]:
(
    alt.Chart(top_keywords)
    .mark_bar()
    .encode(
        x=alt.X(
            "keyword:O",
            title="Keyword",
            sort=alt.SortField("count", order="descending"),
        ),
        y=alt.Y("count:Q", title="Count"),
    )
    .properties(title="Top 15 Keywords")
)

In [ ]:
(
    alt.Chart(top_authors)
    .mark_bar()
    .encode(
        x=alt.X(
            "author_full_name:O",
            title="Author",
            sort=alt.SortField("count", order="descending"),
        ),
        y=alt.Y("count:Q", title="Number of Articles"),
    )
    .properties(title="Top 15 Authors")
)